# Design and implementation



In [40]:
import altair as alt
import pandas as pd
from pathlib import Path
import numpy as np

input_file = Path("simpsons_episodes_clean.csv")
df = pd.read_csv(input_file)

## 1. How have the ratings evolved over time?

In [4]:
chart_1 = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X("season_episode:O", title="Episodes"),
    y=alt.Y("imdb_rating:Q", title="IMDb rating", scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip("season_episode:O", title="Season"),
        alt.Tooltip("imdb_rating:Q", title="Rating", format=".1f")
    ]
).properties(
    title="Evolution of ratings over time",
    height=350
).configure_title(
    fontSize=20
)

chart_1

alt.Chart(...)

In [38]:
chart_2 = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0, orient="top")),
    y=alt.Y("mean(imdb_rating):Q", title="Average IMDb rating", scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip("season:O", title="Season"),
        alt.Tooltip("mean(imdb_rating):Q", title="Avg rating", format=".1f")
    ]
).properties(
    title="Evolution of ratings over time",
    width=600,
    height=350
).configure_title(
    fontSize=20
)

chart_2

alt.Chart(...)

In [39]:
df["rating_group"] = np.select(
    [
        df["imdb_rating"] <= 5,
        (df["imdb_rating"] > 5) & (df["imdb_rating"] < 6),
        (df["imdb_rating"] >= 6) & (df["imdb_rating"] < 7),
        (df["imdb_rating"] >= 7) & (df["imdb_rating"] < 8),
        (df["imdb_rating"] >= 8) & (df["imdb_rating"] < 9),
        df["imdb_rating"] >= 9
    ],
    [
        "<=5",
        "5-5.9",
        "6-6.9",
        "7-7.9",
        "8-8.9",
        ">=9"
    ]
)

base = alt.Chart(df).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0, orient="top")),
    y=alt.Y("number_in_season:O", title="Episodes", sort="ascending")
)

heatmap = base.mark_rect(cornerRadius=6).encode(
    color=alt.Color(
        "rating_group:N",
        title="Rating",
        scale=alt.Scale(
            domain=["<=5", "5-5.9", "6-6.9", "7-7.9", "8-8.9", ">=9"],
            range=["#780000", "#D73939", "#d47336", "#f9ec5f", "#3dbd3d", "#005800"]
        )
    )
)

text = base.mark_text(fontSize=14, fontWeight="bold").encode(
    text=alt.Text("imdb_rating:Q", format=".1f"),
    color=alt.condition(
        (alt.datum.imdb_rating >= 9) | (alt.datum.imdb_rating <= 5.9),
        alt.value("white"),
        alt.value("black")
    )
)

chart = (heatmap + text).properties(
    width=800,
    height=800,
    title="Episode ratings by season"
)

chart

alt.LayerChart(...)

### Design decisions:

## 2. How have the viewers evolved over time? 

In [35]:
chart_3 = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("mean(us_viewers_in_millions):Q", title="Average US viewers in millions", scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip("season:O", title="Season"),
        alt.Tooltip("mean(us_viewers_in_millions):Q", title="Avg US viewers in millions", format=".1f")
    ]
).properties(
    title="Evolution of viewers over time",
    width=600,
    height=350
).configure_title(
    fontSize=20
)

chart_3

alt.Chart(...)

In [70]:
base = alt.Chart(df).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0, orient="top")),
    y=alt.Y("number_in_season:O",title="Episodes", sort="ascending")
)

heatmap = base.mark_rect(cornerRadius=6).encode(
    color=alt.Color(
        "us_viewers_in_millions:Q",
        title="Rating",
        scale=alt.Scale(scheme=alt.SchemeParams(name="redyellowgreen", extent=[0.2, 1]))
    )
)

text = base.mark_text(fontSize=11, fontWeight="bold").encode(
    text=alt.Text("us_viewers_in_millions:Q", format=".1f"),
    color=alt.value("black")
)

chart = (heatmap + text).properties(
    width=800,
    height=800,
    title="Episode viewers by season"
).configure_title(
    fontSize=20
)

chart

alt.LayerChart(...)

### Design decisions:

## 3. Is there a correlation between the gradings and the viewers? 

In [71]:
corr = df["imdb_rating"].corr(df["us_viewers_in_millions"])

chart_3 = alt.Chart(df).mark_circle(size=80, opacity=0.55).encode(
    x=alt.X(
        "imdb_rating:Q",
        title="IMDb rating",
        scale=alt.Scale(domain=[4, 10])
    ),
    y=alt.Y(
        "us_viewers_in_millions:Q",
        title="US viewers (millions)",
        scale=alt.Scale(domainMin=0)
    ),
    tooltip=[
        alt.Tooltip("imdb_rating:Q", title="IMDb rating", format=".1f"),
        alt.Tooltip("us_viewers_in_millions:Q", title="Viewers", format=".2f")
    ]
)

trend = alt.Chart(df).transform_regression(
    "imdb_rating", "us_viewers_in_millions"
).mark_line(size=3, clip=True, color="red").encode(
    x=alt.X("imdb_rating:Q", scale=alt.Scale(domain=[4, 10])),
    y=alt.Y("us_viewers_in_millions:Q", scale=alt.Scale(domainMin=0))
)

corr_text = alt.Chart(
    pd.DataFrame({
        "x": [4.5],
        "y": [32.5],
        "label": [f"Corr = {corr:.3f}"]
    })
).mark_text(
    align="left",
    baseline="top",
    fontSize=14,
    fontWeight="bold"
).encode(
    x="x:Q",
    y="y:Q",
    text="label:N"
)

chart_3 = (chart_3 + trend + corr_text).properties(
    title="Correlation between ratings and viewers",
    width=600,
    height=350
).configure_title(
    fontSize=20
)

chart_3

alt.LayerChart(...)

### Design decisions:

## 4. Are the number of viewers for the episodes related to the weekday they were aired?

In [76]:
weekday_order = ["Monday", "Tuesday", "Wednesday","Thursday", "Friday", "Saturday", "Sunday"]

weekday_map = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

df["weekday"] = df["weekday_num"].map(weekday_map)

chart_4 = alt.Chart(df).mark_boxplot(size=40).encode(
    x=alt.X("weekday:N", sort=weekday_order, title="Weekday aired"),
    y=alt.Y("us_viewers_in_millions:Q", title="US viewers (millions)"),
).properties(
    title="US viewers by weekday aired",
    width=600,
    height=350
).configure_title(
    fontSize=20
)

chart_4

alt.Chart(...)

In [78]:
chart_weekday_counts = alt.Chart(df).mark_bar().encode(
    x=alt.X("weekday:N", sort=weekday_order, title="Weekday aired"),
    y=alt.Y("count():Q", title="Number of episodes"),
    tooltip=[
        alt.Tooltip("weekday:N", title="Weekday"),
        alt.Tooltip("count():Q", title="Episodes")
    ]
).properties(
    title="Number of episodes by weekday",
    width=600,
    height=350
)

chart_weekday_counts

alt.Chart(...)

### Design decisions:
sASA

## 5. Do the seasons number of viewers present any relevant pattern? 


In [79]:
season_viewers = (
    df.groupby("season", as_index=False)
    .agg(
        avg_viewers=("us_viewers_in_millions", "mean"),
        median_viewers=("us_viewers_in_millions", "median"),
        n_episodes=("us_viewers_in_millions", "count")
    )
)

chart_5 = alt.Chart(season_viewers).mark_line(point=True).encode(
    x=alt.X("season:O", title="Season"),
    y=alt.Y("avg_viewers:Q", title="Average US viewers (millions)"),
    tooltip=[
        alt.Tooltip("season:O", title="Season"),
        alt.Tooltip("avg_viewers:Q", title="Average viewers", format=".2f"),
        alt.Tooltip("median_viewers:Q", title="Median viewers", format=".2f"),
        alt.Tooltip("n_episodes:Q", title="Episodes")
    ]
).properties(
    title="Average viewers by season",
    width=700,
    height=350
)

chart_5

alt.Chart(...)

In [80]:
chart_5_box = alt.Chart(df).mark_boxplot(size=20).encode(
    x=alt.X("season:O", title="Season"),
    y=alt.Y("us_viewers_in_millions:Q", title="US viewers (millions)"),
    tooltip=[
        alt.Tooltip("season:O", title="Season")
    ]
).properties(
    title="Viewers distribution by season",
    width=700,
    height=350
)

chart_5_box

alt.Chart(...)

### Design decisions: